## Imports

In [34]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, TextVectorization, Embedding, Dense, SimpleRNN, LSTM, GRU, RNN, SimpleRNNCell
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

In [35]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 10)

## Utils

In [36]:
def build_model(rnn_type="SimpleRNN"):
    model = Sequential()
    
    model.add(Input(shape=(max_len,), dtype="int32"))
    model.add(Embedding(input_dim=max_tokens + 1, output_dim=128))
    
    if rnn_type == "SimpleRNN":
        model.add(SimpleRNN(64))
    elif rnn_type == "LSTM":
        model.add(LSTM(64))
    elif rnn_type == "GRU":
        model.add(GRU(64))
        
    model.add(Dense(64, activation="relu"))
    model.add(Dense(1, activation="sigmoid"))
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
def predict(model, sample_texts, sample_x, sample_labels_true, num_samples, title):
    
    print("Generating predictions...")
    preds_model = model.predict(sample_x, verbose=0)
    
    for i in range(num_samples):
        text_snippet = sample_texts[i].replace('\n', ' ')
        if len(text_snippet) > 80:
            text_snippet = text_snippet[:80] + "..."
            
        true_sentiment = "Positive" if sample_labels_true[i] == 1.0 else "Negative"
        
        model_val = preds_model[i][0]
        
        model_sentiment = "Positive" if model_val > 0.5 else "Negative"
        
        print(f"\n[Sample {i+1}] Text: '{text_snippet}'")
        print(f"  - True Label : {true_sentiment}")
        print(f"  - {title}  : {model_sentiment:<8} (Confidence: {model_val:.4f})")

## Dataload

In [38]:
print("Loading data from CSV...")
df = pd.read_csv("../data/IMDB Dataset.csv")

df.info()

Loading data from CSV...
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 781.4 KB


In [39]:
texts = df['review'].astype(str).tolist()
labels = np.where(
    df['sentiment'].astype(str).str.lower().str.strip() == 'positive', 
    1.0, 
    0.0
).astype(np.float32)

print("Splitting data into train, validation, and test sets...")
train_texts, temp_texts, train_labels, temp_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)
val_texts, test_texts, val_labels, test_labels = train_test_split(temp_texts, temp_labels, test_size=0.5, random_state=42)

Splitting data into train, validation, and test sets...


In [40]:
max_tokens = 1000  
max_len = 100       

vectorize_layer = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_len
)

print("Adapting TextVectorization layer to the training vocabulary...")
vectorize_layer.adapt(train_texts)

print("Pre-vectorizing the text data...")
train_x = vectorize_layer(train_texts)
val_x = vectorize_layer(val_texts)
test_x = vectorize_layer(test_texts)

train_y = tf.constant(train_labels)
val_y = tf.constant(val_labels)
test_y = tf.constant(test_labels)

Adapting TextVectorization layer to the training vocabulary...
Pre-vectorizing the text data...


## Training

In [41]:
epochs_to_run = 5 
batch_size = 256

custom_reviews = [
    "This movie was an absolute triumph. The direction was brilliant and the acting was top notch.",
    "I regret spending money on this. The plot made no sense and it was incredibly boring."
]

custom_reviews_vectorized = vectorize_layer(custom_reviews)

num_samples = 10
sample_texts = test_texts[:num_samples]
sample_labels_true = test_labels[:num_samples]
sample_x = vectorize_layer(sample_texts)

### SimpleRNN

In [42]:
model_rnn = build_model(rnn_type="SimpleRNN")

early_stop_rnn = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
checkpoint_rnn = ModelCheckpoint(filepath="best_model_SimpleRNN.keras", monitor='val_loss', save_best_only=True)

history_rnn = model_rnn.fit(
    x=train_x, y=train_y,
    validation_data=(val_x, val_y),
    epochs=epochs_to_run,
    batch_size=batch_size,
    callbacks=[early_stop_rnn, checkpoint_rnn]
)

print(f"\nEvaluating SimpleRNN on Test Data...")
loss, acc = model_rnn.evaluate(test_x, test_y, batch_size=batch_size, verbose=0)
print(f"Test Accuracy: {acc:.4f}")

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 44ms/step - accuracy: 0.5741 - loss: 0.6651 - val_accuracy: 0.6734 - val_loss: 0.6079
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - accuracy: 0.7458 - loss: 0.5284 - val_accuracy: 0.7508 - val_loss: 0.5479
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 43ms/step - accuracy: 0.7875 - loss: 0.4558 - val_accuracy: 0.7934 - val_loss: 0.4444
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 44ms/step - accuracy: 0.7962 - loss: 0.4438 - val_accuracy: 0.7868 - val_loss: 0.4630
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 43ms/step - accuracy: 0.8100 - loss: 0.4185 - val_accuracy: 0.7906 - val_loss: 0.4603

Evaluating SimpleRNN on Test Data...
Test Accuracy: 0.7932


In [43]:
print(f"\nTest samples predictions:")
predict(model_rnn, sample_texts, sample_x, sample_labels_true, num_samples, title="SimpleRNN Prediction")

print(f"\nCustom Predictions:")
predictions = model_rnn.predict(custom_reviews_vectorized, verbose=0)
for review, pred in zip(custom_reviews, predictions):
    print(f"Review: '{review[:50]}...' -> {'Positive' if pred[0] > 0.5 else 'Negative'} ({pred[0]:.4f})")


Test samples predictions:
Generating predictions...

[Sample 1] TEXT: 'the tortuous emotional impact is degrading, whether adult or adolescent the pers...'
  > True Label : Negative
  > SimpleRNN Prediction  : Negative (Confidence: 0.0398)

[Sample 2] TEXT: 'Anyone who knows anything about evolution wouldn't even need to see the film to ...'
  > True Label : Negative
  > SimpleRNN Prediction  : Negative (Confidence: 0.0433)

[Sample 3] TEXT: 'I'm glad I rented this movie for one reason: its shortcomings made me want to re...'
  > True Label : Negative
  > SimpleRNN Prediction  : Positive (Confidence: 0.5683)

[Sample 4] TEXT: 'Yes, the votes are in. This film may very well be the Plan 9 From Outer Space fo...'
  > True Label : Negative
  > SimpleRNN Prediction  : Positive (Confidence: 0.8109)

[Sample 5] TEXT: 'This mini-series is actually more entertaining than some others with much bigger...'
  > True Label : Negative
  > SimpleRNN Prediction  : Positive (Confidence: 0.8220)

[Sampl

### LSTM

In [44]:
model_lstm = build_model(rnn_type="LSTM")

early_stop_lstm = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
checkpoint_lstm = ModelCheckpoint(filepath="best_model_LSTM.keras", monitor='val_loss', save_best_only=True)

history_lstm = model_lstm.fit(
    x=train_x, y=train_y,
    validation_data=(val_x, val_y),
    epochs=epochs_to_run,
    batch_size=batch_size,
    callbacks=[early_stop_lstm, checkpoint_lstm]
)

print(f"\nEvaluating LSTM on Test Data...")
loss, acc = model_lstm.evaluate(test_x, test_y, batch_size=batch_size, verbose=0)
print(f"Test Accuracy: {acc:.4f}")

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 16s 90ms/step - accuracy: 0.6975 - loss: 0.5529 - val_accuracy: 0.7800 - val_loss: 0.4724
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 13s 81ms/step - accuracy: 0.7904 - loss: 0.4452 - val_accuracy: 0.7978 - val_loss: 0.4316
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 15s 95ms/step - accuracy: 0.8046 - loss: 0.4160 - val_accuracy: 0.8058 - val_loss: 0.4298
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 23s 110ms/step - accuracy: 0.8088 - loss: 0.4099 - val_accuracy: 0.7910 - val_loss: 0.4299
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 19s 99ms/step - accuracy: 0.8177 - loss: 0.3923 - val_accuracy: 0.8124 - val_loss: 0.4034

Evaluating LSTM on Test Data...
Test Accuracy: 0.8124


In [45]:
print(f"\nTest samples predictions:")
predict(model_lstm, sample_texts, sample_x, sample_labels_true, num_samples, title="LSTM Prediction")

print(f"\nCustom Predictions:")
predictions = model_lstm.predict(custom_reviews_vectorized, verbose=0)
for review, pred in zip(custom_reviews, predictions):
    print(f"Review: '{review[:50]}...' -> {'Positive' if pred[0] > 0.5 else 'Negative'} ({pred[0]:.4f})")


Test samples predictions:
Generating predictions...

[Sample 1] TEXT: 'the tortuous emotional impact is degrading, whether adult or adolescent the pers...'
  > True Label : Negative
  > LSTM Prediction  : Negative (Confidence: 0.0193)

[Sample 2] TEXT: 'Anyone who knows anything about evolution wouldn't even need to see the film to ...'
  > True Label : Negative
  > LSTM Prediction  : Negative (Confidence: 0.1087)

[Sample 3] TEXT: 'I'm glad I rented this movie for one reason: its shortcomings made me want to re...'
  > True Label : Negative
  > LSTM Prediction  : Negative (Confidence: 0.0528)

[Sample 4] TEXT: 'Yes, the votes are in. This film may very well be the Plan 9 From Outer Space fo...'
  > True Label : Negative
  > LSTM Prediction  : Positive (Confidence: 0.7482)

[Sample 5] TEXT: 'This mini-series is actually more entertaining than some others with much bigger...'
  > True Label : Negative
  > LSTM Prediction  : Positive (Confidence: 0.5285)

[Sample 6] TEXT: 'For those wit

### GRU

In [46]:
model_gru = build_model(rnn_type="GRU")

early_stop_gru = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
checkpoint_gru = ModelCheckpoint(filepath="best_model_GRU.keras", monitor='val_loss', save_best_only=True)

history_gru = model_gru.fit(
    x=train_x, y=train_y,
    validation_data=(val_x, val_y),
    epochs=epochs_to_run,
    batch_size=batch_size,
    callbacks=[early_stop_gru, checkpoint_gru]
)

print(f"\nEvaluating GRU on Test Data...")
loss, acc = model_gru.evaluate(test_x, test_y, batch_size=batch_size, verbose=0)
print(f"Test Accuracy: {acc:.4f}")

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 18s 101ms/step - accuracy: 0.5951 - loss: 0.6625 - val_accuracy: 0.7134 - val_loss: 0.5588
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 18s 114ms/step - accuracy: 0.7729 - loss: 0.4855 - val_accuracy: 0.7922 - val_loss: 0.4473
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 15s 97ms/step - accuracy: 0.8044 - loss: 0.4268 - val_accuracy: 0.8082 - val_loss: 0.4159
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 14s 86ms/step - accuracy: 0.8133 - loss: 0.4050 - val_accuracy: 0.8096 - val_loss: 0.4093
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 16s 102ms/step - accuracy: 0.8217 - loss: 0.3894 - val_accuracy: 0.8150 - val_loss: 0.3986

Evaluating GRU on Test Data...
Test Accuracy: 0.8144


In [47]:
print(f"\nTest samples predictions:")
predict(model_gru, sample_texts, sample_x, sample_labels_true, num_samples, title="GRU Prediction")

print(f"\nCustom Predictions:")
predictions = model_gru.predict(custom_reviews_vectorized, verbose=0)
for review, pred in zip(custom_reviews, predictions):
    print(f"Review: '{review[:50]}...' -> {'Positive' if pred[0] > 0.5 else 'Negative'} ({pred[0]:.4f})")


Test samples predictions:
Generating predictions...

[Sample 1] TEXT: 'the tortuous emotional impact is degrading, whether adult or adolescent the pers...'
  > True Label : Negative
  > GRU Prediction  : Negative (Confidence: 0.0561)

[Sample 2] TEXT: 'Anyone who knows anything about evolution wouldn't even need to see the film to ...'
  > True Label : Negative
  > GRU Prediction  : Negative (Confidence: 0.0456)

[Sample 3] TEXT: 'I'm glad I rented this movie for one reason: its shortcomings made me want to re...'
  > True Label : Negative
  > GRU Prediction  : Negative (Confidence: 0.0960)

[Sample 4] TEXT: 'Yes, the votes are in. This film may very well be the Plan 9 From Outer Space fo...'
  > True Label : Negative
  > GRU Prediction  : Positive (Confidence: 0.7537)

[Sample 5] TEXT: 'This mini-series is actually more entertaining than some others with much bigger...'
  > True Label : Negative
  > GRU Prediction  : Positive (Confidence: 0.5660)

[Sample 6] TEXT: 'For those with acc